In [1]:
import numpy as np
import pandas as pd
from numpy.linalg import norm
rng = np.random.default_rng(42)

# -----------------------------
# 1) Create a toy rating dataset
# -----------------------------
n_users, n_items = 10, 50
density = 0.25               # ~25% of user–item pairs have ratings
rating_scale = (1, 5)        # 1..5 stars

# Start with NaNs = missing ratings
R = np.full((n_users, n_items), np.nan, dtype=float)

# Randomly place ratings with given density
mask = rng.random((n_users, n_items)) < density
R[mask] = rng.integers(rating_scale[0], rating_scale[1] + 1, size=mask.sum())

users = [f"u{u:02d}" for u in range(n_users)]
items = [f"i{i:02d}" for i in range(n_items)]
ratings_df = pd.DataFrame(R, index=users, columns=items)

print("Ratings (NaN = missing). A few rows:")
print(ratings_df.iloc[:5, :10])


# -----------------------------------------------
# 2) Adjusted-cosine item–item similarity (center
#    each user's ratings by that user's mean)
# -----------------------------------------------

def adjusted_cosine_item_similarity(R_nan: np.ndarray) -> np.ndarray:
    """
    R_nan: shape (n_users, n_items), NaN = missing
    Returns S: item-item similarity matrix (n_items, n_items)
    """
    R = R_nan.copy()
    # Compute user means on observed ratings only
    user_means = np.nanmean(R, axis=1)          # shape (n_users,)
    # Center by user means where rating is present
    for u in range(R.shape[0]):
        idx = ~np.isnan(R[u])
        R[u, idx] = R[u, idx] - user_means[u]

    # Replace NaNs with 0 for cosine calculation (missing -> 0 contribution)
    R_centered = np.nan_to_num(R, nan=0.0)

    # Item vectors are columns
    # Cosine similarity S_ij = (x_i · x_j) / (||x_i|| * ||x_j||)
    X = R_centered  # shape (n_users, n_items)
    G = X.T @ X     # Gram matrix (n_items, n_items)
    norms = np.sqrt(np.diag(G))
    denom = np.outer(norms, norms)
    # Avoid division by zero
    denom[denom == 0] = 1e-12
    S = G / denom

    # Remove tiny numerical noise off the diagonal
    np.fill_diagonal(S, 1.0)
    return S

S = adjusted_cosine_item_similarity(ratings_df.values)
pd.DataFrame(S, index=items, columns=items).iloc[:5, :5]


# -------------------------------------------
# 3) Predict & recommend for a specific user
# -------------------------------------------

def recommend_item_based_for_user(user_idx: int, R_nan: np.ndarray, S: np.ndarray, top_k_neighbors=20, top_n=10):
    """
    Predict ratings for a single user on their UNRATED items using item-based CF with adjusted-cosine.
    Returns a list of (item_index, score) sorted by descending score.
    """
    R = R_nan
    user_ratings = R[user_idx]               # shape (n_items,)
    rated_mask = ~np.isnan(user_ratings)
    unrated_mask = np.isnan(user_ratings)

    if rated_mask.sum() == 0:
        return []  # nothing to base predictions on

    # User mean over observed ratings
    user_mean = np.nanmean(user_ratings)

    # Residuals (r_uj - mean_u) for rated items only
    residuals = np.zeros_like(user_ratings)
    residuals[rated_mask] = user_ratings[rated_mask] - user_mean

    scores = np.full(R.shape[1], -np.inf)

    # For each target item i (unrated), aggregate over similar rated items j
    for i in np.where(unrated_mask)[0]:
        # Keep similarities only to items j that the user has rated
        sims = S[i].copy()
        sims[~rated_mask] = 0.0

        # Select top-k neighbors N(i; u) with largest |S_ij|
        nbr_idx = np.argsort(-np.abs(sims))[:top_k_neighbors]
        sim_nbrs = sims[nbr_idx]

        denom = np.sum(np.abs(sim_nbrs))
        if denom == 0:
            continue

        # Predict rating: r_hat_ui = mean_u + sum(S_ij * (r_uj - mean_u)) / sum(|S_ij|)
        scores[i] = user_mean + np.dot(sim_nbrs, residuals[nbr_idx]) / denom

    # Rank by predicted score (descending)
    ranked_items = np.argsort(-scores)
    recs = [(i, scores[i]) for i in ranked_items if np.isfinite(scores[i])]
    return recs[:top_n]

# Example: recommend for user u03
user_name = "u03"
user_idx = users.index(user_name)
recs = recommend_item_based_for_user(user_idx, ratings_df.values, S, top_k_neighbors=15, top_n=8)

print(f"Top recommendations for {user_name}:")
for i, s in recs:
    print(f"{items[i]:>4}  predicted_rating={s:.3f}")


Ratings (NaN = missing). A few rows:
     i00  i01  i02  i03  i04  i05  i06  i07  i08  i09
u00  NaN  NaN  NaN  NaN  5.0  NaN  NaN  NaN  4.0  NaN
u01  4.0  2.0  NaN  NaN  NaN  NaN  NaN  NaN  2.0  3.0
u02  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1.0  NaN
u03  NaN  NaN  1.0  NaN  3.0  NaN  NaN  NaN  NaN  NaN
u04  NaN  NaN  NaN  3.0  2.0  3.0  5.0  NaN  1.0  NaN
Top recommendations for u03:
 i19  predicted_rating=4.261
 i30  predicted_rating=4.047
 i31  predicted_rating=4.038
 i09  predicted_rating=3.974
 i40  predicted_rating=3.943
 i47  predicted_rating=3.933
 i43  predicted_rating=3.877
 i17  predicted_rating=3.821


In [13]:
np.nanmean(ratings_df.values, axis = 1)

array([3.63636364, 2.73333333, 2.54545455, 3.        , 3.14285714,
       3.23809524, 3.66666667, 3.        , 2.55555556, 3.45454545])